<a href="https://colab.research.google.com/github/BikashKumarSethy/BikashKumarSethy/blob/main/MultiAgentFinanceEnhancerLvl_1LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install langchain langgraph tavily-python openai yfinance pandas langchain_openai

In [16]:
from tavily import TavilyClient

TAVILY_API_KEY = "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"

tavily = TavilyClient(api_key=TAVILY_API_KEY)

# **LLM Setup**

In [17]:
import os
from openai import OpenAI

OPENROUTER_API_KEY = "sk-or-v1-abrakadabra"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [18]:
def llm_call(prompt):

    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

**Agent 1 — Intent Detection Agent**

In [19]:
def intent_agent(state):

    query = state["query"]

    prompt = f"""
    Classify the user query into one of the following:

    fraud
    market
    portfolio
    compliance
    general

    Query: {query}

    Return only one label.
    """

    intent = llm_call(prompt)

    # Ensure the original query is carried forward in the state
    return {"query": query, "intent": intent.strip()}


**Agent 2 — Market Intelligence Agent (Tavily)**

In [20]:
def market_agent(state):

    query = state["query"]

    results = tavily.search(
        query=query,
        search_depth="advanced",
        max_results=5
    )

    news = []

    for r in results["results"]:
        news.append(r["content"])

    return {"market_news": news}

**Agent 3 — Portfolio Recommendation Agent**

In [21]:
import yfinance as yf

def portfolio_agent(state):

    stocks = ["AAPL","MSFT","NVDA"]

    recommendations = {}

    for s in stocks:

        data = yf.Ticker(s)

        info = data.info

        recommendations[s] = {
            "price": info.get("currentPrice"),
            "sector": info.get("sector")
        }

    return {"portfolio": recommendations}

**Agent 4 — Fraud Detection Agent**

In [22]:
def fraud_agent(state):

    transaction = state.get("transaction",{})

    amount = transaction.get("amount",0)
    location = transaction.get("location","")

    risk = "low"

    if amount > 100000:
        risk = "high"

    return {
        "fraud_risk": risk
    }

**Agent 5 — Compliance Agent**

In [23]:
def compliance_agent(state):

    prompt = f"""
    Analyze this financial activity for regulatory or compliance risks.

    Activity:
    {state}

    Provide a short compliance assessment.
    """

    result = llm_call(prompt)

    return {"compliance_analysis": result}

**Router Function**

In [24]:
def router(state):

    intent = state["intent"]

    if "fraud" in intent:
        return "fraud"

    if "market" in intent:
        return "market"

    if "portfolio" in intent:
        return "portfolio"

    return "general"

**LangGraph Construction**

In [25]:
from langgraph.graph import StateGraph

workflow = StateGraph(dict)

workflow.add_node("intent", intent_agent)
workflow.add_node("fraud", fraud_agent)
workflow.add_node("market", market_agent)
workflow.add_node("portfolio", portfolio_agent)
workflow.add_node("compliance", compliance_agent)

workflow.set_entry_point("intent")

workflow.add_conditional_edges(
    "intent",
    router,
    {
        "fraud": "fraud",
        "market": "market",
        "portfolio": "portfolio"
    }
)

workflow.add_edge("fraud","compliance")
workflow.add_edge("market","compliance")
workflow.add_edge("portfolio","compliance")

workflow.set_finish_point("compliance")

app = workflow.compile()

## **Running the System**

In [26]:
query = {
    "query":"What is the market outlook for Nvidia stock?"
}

result = app.invoke(query)

print(result)

{'compliance_analysis': "Based on the financial activity provided, there are several compliance risks that should be considered:\n\n1. Insider Trading: With detailed information on NVIDIA's financial performance, stock price forecasts, and analyst recommendations, there is a risk of insider trading if employees or individuals with access to this information use it for personal gain.\n\n2. Market Manipulation: The detailed analysis of NVIDIA's stock price, technical indicators, and analyst forecasts could potentially be used to manipulate the market by spreading false information or creating artificial demand.\n\n3. Regulatory Disclosure Requirements: The activity includes a significant amount of financial data and forecasts, which may trigger regulatory disclosure requirements to ensure transparency and accuracy in reporting.\n\n4. Conflict of Interest: Analyst reports and price targets provided in the activity may raise concerns about potential conflicts of interest if the analysts ha

**Front End using Gradio**

In [27]:
!pip install gradio

**Create a Query Handler Function**

In [28]:
def financial_ai_system(user_query):

    try:

        state = {
            "query": user_query
        }

        result = app.invoke(state)

        response = ""

        if "answer" in result:
            response += f"\nAnswer:\n{result['answer']}\n"

        if "market_news" in result:
            response += "\nMarket Insights:\n"
            for news in result["market_news"][:3]:
                response += f"- {news}\n"

        if "portfolio" in result:
            response += "\nPortfolio Suggestions:\n"
            for stock,data in result["portfolio"].items():
                response += f"{stock}: Price {data['price']}\n"

        if "fraud_risk" in result:
            response += f"\nFraud Risk Level: {result['fraud_risk']}\n"

        if "compliance_analysis" in result:
            response += f"\nCompliance Check:\n{result['compliance_analysis']}\n"

        return response

    except Exception as e:
        return f"Error: {str(e)}"

In [30]:
import gradio as gr

interface = gr.Interface(

    fn=financial_ai_system,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask about fraud detection, markets, portfolio, compliance..."
    ),

    outputs="text",

    title="AI Financial Intelligence Platform",

    description="""
    Multi-Agent FinTech AI System powered by LangGraph

    Capabilities:
    • Fraud Detection
    • Market Intelligence
    • Portfolio Recommendation
    • Compliance Analysis
    • Financial Q&A
    """
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f9fdd6579e93bf882f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
